# Pipeline 11: Social Content Mix Efficiency

## Which content mix produces the strongest donation response?

**Notebook:** `social-content-mix-efficiency.ipynb`  
**Domain:** Outreach and Communications  
**Purpose:** compare content, channel, and timing choices to see which post mixes are most efficient at generating donation value and referrals.

---

## 1. Problem framing

### Business question
Which combinations of platform, post style, tone, and timing are most associated with stronger fundraising performance?

### Predictive and explanatory goals
- **Explanatory model:** estimate how content features relate to donation value.
- **Predictive model:** predict whether a post will land in the high-referral class before it is published.

### Who uses this
- **Founders** who need content guidance without a marketing team
- **Volunteer social media managers** who need a simple pre-publish advisor
- **Outreach leads** who want to compare channels and creative styles

---

## 2. Data and feature engineering

### Core sources
- `social_media_posts`
- `donations`

### Feature examples
- Platform
- Post type and media type
- Sentiment tone and content topic
- Call-to-action flags
- Resident-story flag
- Boosted versus organic promotion
- Hour of posting
- Hashtag count, caption length, engagement, impressions, and reach
- Donation referral rollups joined back to the original post

### Target variables
- **Regression:** log donation value
- **Classification:** high-referral post class

---

## 3. Modeling approach

### Explanatory track
A linear regression model estimates which content features are associated with larger donation values.

### Predictive track
A random forest classifier predicts whether a post belongs to the highest referral quartile, compared against a dummy baseline.

### Validation strategy
- Train/test split plus cross-validation
- Permutation importance to rank the most useful post features
- Coefficient inspection for the explanatory model

---

## 4. What the dashboard can show

### Useful insights
- Which platform performs best for donation conversion
- Whether resident-story posts outperform other content types
- Whether boosted posts or strong calls to action matter most
- Which combinations of timing and tone should be used more often

### Decision output
This notebook supports a pre-publish content advisor that helps the team choose the strongest post mix before a campaign goes live.

---

## 5. Caveats
The notebook describes associations from observed posts. It should be presented as a guidance tool, not as proof that a single content choice will always cause higher donations.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import f1_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent, Path.cwd().parent / 'ml-pipelines']:
    if (candidate / 'data_loader.py').exists():
        sys.path.insert(0, str(candidate))
        break
from data_loader import load_table

posts = load_table('social_media_posts')
donations = load_table('donations')
donations = donations[donations['referral_post_id'].notna()].copy()
donations['referral_post_id'] = donations['referral_post_id'].astype(int)
donation_rollup = donations.groupby('referral_post_id').agg(
    referred_count=('donation_id', 'count'),
    referred_amount=('amount', 'sum')
).reset_index()
df = posts.merge(donation_rollup, left_on='post_id', right_on='referral_post_id', how='left')
df[['referred_count', 'referred_amount']] = df[['referred_count', 'referred_amount']].fillna(0)
df['target_value'] = np.log1p(df['estimated_donation_value_php'].clip(lower=0))
df['high_referral'] = (df['donation_referrals'] >= df['donation_referrals'].quantile(0.75)).astype(int)

feature_cols = [
    'platform', 'post_type', 'media_type', 'sentiment_tone', 'content_topic',
    'has_call_to_action', 'features_resident_story', 'is_boosted', 'post_hour',
    'num_hashtags', 'caption_length', 'engagement_rate', 'impressions', 'reach'
]
X = df[feature_cols].copy()
y_reg = df['target_value']
y_clf = df['high_referral']

cat_cols = [c for c in feature_cols if X[c].dtype == 'object' or X[c].dtype == 'bool']
num_cols = [c for c in feature_cols if c not in cat_cols]
prep = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

Xtr, Xte, ytr_reg, yte_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
lin = Pipeline([('prep', prep), ('model', LinearRegression())])
lin.fit(Xtr, ytr_reg)
pred_reg = lin.predict(Xte)
print('Explanatory model R2:', round(r2_score(yte_reg, pred_reg), 3))
print('Explanatory model MAE:', round(mean_absolute_error(yte_reg, pred_reg), 3))

Xtrc, Xtec, ytrc, ytec = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
baseline = Pipeline([('prep', prep), ('model', DummyClassifier(strategy='prior'))])
rf = Pipeline([('prep', prep), ('model', RandomForestClassifier(n_estimators=300, random_state=42, min_samples_leaf=4))])
baseline.fit(Xtrc, ytrc)
rf.fit(Xtrc, ytrc)
proba_base = baseline.predict_proba(Xtec)[:, 1]
proba_rf = rf.predict_proba(Xtec)[:, 1]
pred_rf = (proba_rf >= 0.5).astype(int)
print('Predictive baseline AUC:', round(roc_auc_score(ytec, proba_base), 3))
print('Predictive RF AUC:', round(roc_auc_score(ytec, proba_rf), 3))
print('Predictive RF F1:', round(f1_score(ytec, pred_rf), 3))

cv = cross_validate(rf, X, y_clf, cv=5, scoring=['roc_auc', 'f1'])
print('CV ROC-AUC mean:', round(float(cv['test_roc_auc'].mean()), 3), 'std:', round(float(cv['test_roc_auc'].std()), 3))

perm = permutation_importance(rf, Xtec, ytec, n_repeats=8, random_state=42, scoring='roc_auc')
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(10)
print('\nTop predictive features:')
print(imp.round(4).to_string())

lin_feature_names = lin.named_steps['prep'].get_feature_names_out()
coef_values = np.ravel(lin.named_steps['model'].coef_)
usable = min(len(coef_values), len(lin_feature_names))
coef = pd.Series(coef_values[:usable], index=lin_feature_names[:usable]).sort_values(key=lambda s: s.abs(), ascending=False).head(10)
print('\nTop explanatory coefficients:')
print(coef.round(4).to_string())

print('\nDecision recommendations: prioritize high-impact content mixes and validate with A/B scheduling experiments.')


ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: 'WhatsApp'

In [ ]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent, Path.cwd().parent / 'ml-pipelines']:
    if (candidate / 'trend_eval_helpers.py').exists():
        sys.path.insert(0, str(candidate))
        break
from trend_eval_helpers import print_calibration_bins, print_threshold_scan, bootstrap_linear_coefs, fairness_binary, fairness_regression_mae

print("\n=== Evaluation artifacts ===")
if ytec.nunique() >= 2:
    proba = rf.predict_proba(Xtec)[:, 1]
    print_calibration_bins(ytec.values, proba)
    print_threshold_scan(ytec.values, proba)
    fairness_binary(rf, Xtec, ytec, df.loc[Xtec.index], ['platform'], min_n=15)
bootstrap_linear_coefs(lin, Xtr, ytr_reg, n_boot=150, top_k=8)
